In [7]:
import pandas as pd
import numpy as np

# ======================
# 1. LOAD DATA
# ======================
cols = [
    "age","workclass","fnlwgt","education","education_num",
    "marital_status","occupation","relationship","race",
    "sex","capital_gain","capital_loss","hours_per_week",
    "native_country","income"
]

df = pd.read_csv("dataset_p1/5adultCenso/adult_data.csv", names=cols, na_values=" ?", skipinitialspace=True)

print("Shape:", df.shape)
print(df.head())

# ======================
# 2. MISSING VALUES
# ======================
print("\nMissing:\n", df.isnull().sum())

# Eliminar filas con missing (estrategia simple y efectiva aquí)
df = df.dropna()

# ======================
# 3. LIMPIEZA TARGET
# ======================
df["income"] = df["income"].str.replace(".", "", regex=False)

df["target"] = df["income"].map({
    "<=50K": 0,
    ">50K": 1
})

df = df.drop(columns=["income"])

# ======================
# 4. FEATURE ENGINEERING
# ======================

# Agrupar categorías raras (muy importante)
for col in df.select_dtypes(include=['object']).columns:
    counts = df[col].value_counts()
    rare = counts[counts < 100].index
    df[col] = df[col].replace(rare, "Other")

# ======================
# 5. ENCODING
# ======================
df = pd.get_dummies(df, drop_first=True)

# ======================
# 6. FEATURES / TARGET
# ======================
X = df.drop(columns=["target"])
y = df["target"]

print("Final shape:", X.shape)
print("Class balance:\n", y.value_counts(normalize=True))
print("Features:\n", X.columns)

Shape: (32561, 15)
   age         workclass  fnlwgt  education  education_num  \
0   39         State-gov   77516  Bachelors             13   
1   50  Self-emp-not-inc   83311  Bachelors             13   
2   38           Private  215646    HS-grad              9   
3   53           Private  234721       11th              7   
4   28           Private  338409  Bachelors             13   

       marital_status         occupation   relationship   race     sex  \
0       Never-married       Adm-clerical  Not-in-family  White    Male   
1  Married-civ-spouse    Exec-managerial        Husband  White    Male   
2            Divorced  Handlers-cleaners  Not-in-family  White    Male   
3  Married-civ-spouse  Handlers-cleaners        Husband  Black    Male   
4  Married-civ-spouse     Prof-specialty           Wife  Black  Female   

   capital_gain  capital_loss  hours_per_week native_country income  
0          2174             0              40  United-States  <=50K  
1             0        

C:\Users\josia\AppData\Local\Temp\ipykernel_1052\2335016016.py:44: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include=['object']).columns:


In [10]:
from sklearn.preprocessing import StandardScaler

num_cols = X.select_dtypes(include=np.number).columns

scaler = StandardScaler()
X[num_cols] = scaler.fit_transform(X[num_cols])

# manejo de desbalance
from sklearn.utils.class_weight import compute_class_weight

weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y),
    y=y
)